# Exercise 11 - Drone Radiation Data Normalization


## Data Model Design

### GPS_DEVICE (SCD Type 2)
- GPS_DEVICE_KEY, GPS_UNIT_NUMBER, CALIBRATION_PRECISION, EFF_START_TS, EFF_END_TS
- One row per GPS calibration period
- EFF_END_TS = NULL for current calibration

### DETECTOR_DEVICE (SCD Type 2) 
- DETECTOR_DEVICE_KEY, DETECTOR_TYPE, DETECTOR_UNIT_NUMBER, CALIBRATION_PRECISION, EFF_START_TS, EFF_END_TS
- All three detector types (GAMMA, CESIUM_137, THORIUM_232) in one table
- One row per detector calibration period
- EFF_END_TS = NULL for current calibration

### MEASUREMENT (Fact)
- COLLECTION_TIMESTAMP, GPS_LAT, GPS_LNG, GAMMA_LEVEL, CESIUM_137_LEVEL, THORIUM_232_LEVEL
- GPS_DEVICE_KEY, GAMMA_DETECTOR_KEY, CESIUM_DETECTOR_KEY, THORIUM_DETECTOR_KEY
- One row per measurement event
- FKs link to the calibration active at measurement time

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load the raw data
SOURCE = "s3a://data5035-spring26/drone_data.json"
raw_df = spark.read.json(SOURCE)

print(f"Loaded {raw_df.count()} rows")

## GPS Dimension

In [0]:
# Get unique GPS calibration events
gps_cals = raw_df.select(
    "GPS_UNIT_NUMBER",
    F.col("GPS_UNIT_CALIBRATION_TIMESTAMP").alias("cal_ts"),
    F.col("GPS_UNIT_CALIBRATION_PRECISION").alias("precision")
).distinct().orderBy("GPS_UNIT_NUMBER", "cal_ts")

# Add effective dates using window function
w = Window.partitionBy("GPS_UNIT_NUMBER").orderBy("cal_ts")
gps_dimension = gps_cals.withColumn("next_cal", F.lead("cal_ts").over(w)) \
    .withColumn("gps_key", F.monotonically_increasing_id()) \
    .select(
        F.col("gps_key").alias("GPS_DEVICE_KEY"),
        "GPS_UNIT_NUMBER",
        F.col("precision").alias("CALIBRATION_PRECISION"),
        F.col("cal_ts").alias("EFF_START_TS"),
        F.col("next_cal").alias("EFF_END_TS")
    )

print(f"GPS dimension: {gps_dimension.count()} rows")
display(gps_dimension)

## Detector Dimension

In [0]:
# Gamma detector calibrations
gamma_cals = raw_df.select(
    F.lit("GAMMA").alias("DETECTOR_TYPE"),
    F.col("GAMMA_DETECTOR_UNIT_NUMBER").alias("unit_num"),
    F.col("GAMMA_DETECTOR_CALIBRATION_TIMESTAMP").alias("cal_ts"),
    F.col("GAMMA_DETECTOR_CALIBRATION_PRECISION").alias("precision")
).distinct()

# Cesium detector calibrations
cesium_cals = raw_df.select(
    F.lit("CESIUM_137").alias("DETECTOR_TYPE"),
    F.col("CESIUM_137_DETECTOR_UNIT_NUMBER").alias("unit_num"),
    F.col("CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP").alias("cal_ts"),
    F.col("CESIUM_137_DETECTOR_CALIBRATION_PRECISION").alias("precision")
).distinct()

# Thorium detector calibrations
thorium_cals = raw_df.select(
    F.lit("THORIUM_232").alias("DETECTOR_TYPE"),
    F.col("THORIUM_232_DETECTOR_UNIT_NUMBER").alias("unit_num"),
    F.col("THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP").alias("cal_ts"),
    F.col("THORIUM_232_DETECTOR_CALIBRATION_PRECISION").alias("precision")
).distinct()

# Combine all detector calibrations
detector_cals = gamma_cals.union(cesium_cals).union(thorium_cals) \
    .orderBy("DETECTOR_TYPE", "unit_num", "cal_ts")

print(f"All detector calibrations: {detector_cals.count()}")

In [0]:
# Add effective dates for detector dimension
detector_window = Window.partitionBy("DETECTOR_TYPE", "unit_num").orderBy("cal_ts")
detector_dimension = detector_cals.withColumn("next_cal", F.lead("cal_ts").over(detector_window)) \
    .withColumn("detector_key", F.monotonically_increasing_id()) \
    .select(
        F.col("detector_key").alias("DETECTOR_DEVICE_KEY"),
        "DETECTOR_TYPE",
        F.col("unit_num").alias("DETECTOR_UNIT_NUMBER"),
        F.col("precision").alias("CALIBRATION_PRECISION"),
        F.col("cal_ts").alias("EFF_START_TS"),
        F.col("next_cal").alias("EFF_END_TS")
    )

print(f"Detector dimension: {detector_dimension.count()} rows")
display(detector_dimension)

## Measurement Fact Table

In [0]:
# Start with the measurement data
measurements = raw_df.select(
    "COLLECTION_TIMESTAMP",
    "GPS_LAT", "GPS_LNG",
    "GAMMA_LEVEL", "CESIUM_137_LEVEL", "THORIUM_232_LEVEL",
    "GPS_UNIT_NUMBER", "GPS_UNIT_CALIBRATION_TIMESTAMP",
    "GAMMA_DETECTOR_UNIT_NUMBER", "GAMMA_DETECTOR_CALIBRATION_TIMESTAMP",
    "CESIUM_137_DETECTOR_UNIT_NUMBER", "CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP",
    "THORIUM_232_DETECTOR_UNIT_NUMBER", "THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP"
)

# Join GPS device key
fact = measurements.join(
    gps_dimension.select("GPS_DEVICE_KEY", "GPS_UNIT_NUMBER", "EFF_START_TS"),
    (measurements.GPS_UNIT_NUMBER == gps_dimension.GPS_UNIT_NUMBER) & 
    (measurements.GPS_UNIT_CALIBRATION_TIMESTAMP == gps_dimension.EFF_START_TS),
    "left"
).drop(gps_dimension.GPS_UNIT_NUMBER).drop(gps_dimension.EFF_START_TS)

print(f"After GPS join: {fact.count()}")

In [0]:
# Join gamma detector key
gamma_dim = detector_dimension.filter(F.col("DETECTOR_TYPE") == "GAMMA") \
    .select(F.col("DETECTOR_DEVICE_KEY").alias("GAMMA_DETECTOR_KEY"), 
            "DETECTOR_UNIT_NUMBER", "EFF_START_TS")

fact = fact.join(
    gamma_dim,
    (fact.GAMMA_DETECTOR_UNIT_NUMBER == gamma_dim.DETECTOR_UNIT_NUMBER) &
    (fact.GAMMA_DETECTOR_CALIBRATION_TIMESTAMP == gamma_dim.EFF_START_TS),
    "left"
).drop(gamma_dim.DETECTOR_UNIT_NUMBER).drop(gamma_dim.EFF_START_TS)

# Join cesium detector key
cesium_dim = detector_dimension.filter(F.col("DETECTOR_TYPE") == "CESIUM_137") \
    .select(F.col("DETECTOR_DEVICE_KEY").alias("CESIUM_DETECTOR_KEY"),
            "DETECTOR_UNIT_NUMBER", "EFF_START_TS")

fact = fact.join(
    cesium_dim,
    (fact.CESIUM_137_DETECTOR_UNIT_NUMBER == cesium_dim.DETECTOR_UNIT_NUMBER) &
    (fact.CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP == cesium_dim.EFF_START_TS),
    "left"
).drop(cesium_dim.DETECTOR_UNIT_NUMBER).drop(cesium_dim.EFF_START_TS)

# Join thorium detector key
thorium_dim = detector_dimension.filter(F.col("DETECTOR_TYPE") == "THORIUM_232") \
    .select(F.col("DETECTOR_DEVICE_KEY").alias("THORIUM_DETECTOR_KEY"),
            "DETECTOR_UNIT_NUMBER", "EFF_START_TS")

fact_measurements = fact.join(
    thorium_dim,
    (fact.THORIUM_232_DETECTOR_UNIT_NUMBER == thorium_dim.DETECTOR_UNIT_NUMBER) &
    (fact.THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP == thorium_dim.EFF_START_TS),
    "left"
).drop(thorium_dim.DETECTOR_UNIT_NUMBER).drop(thorium_dim.EFF_START_TS)

print(f"Final fact table: {fact_measurements.count()}")

In [0]:
# Clean up final fact table
fact_final = fact_measurements.select(
    "COLLECTION_TIMESTAMP",
    "GPS_LAT", "GPS_LNG",
    "GAMMA_LEVEL", "CESIUM_137_LEVEL", "THORIUM_232_LEVEL",
    "GPS_DEVICE_KEY",
    "GAMMA_DETECTOR_KEY", "CESIUM_DETECTOR_KEY", "THORIUM_DETECTOR_KEY"
)

display(fact_final.limit(10))

## Data Quality Check

In [0]:
# Check for missing keys
print(f"Missing GPS keys: {fact_final.filter(F.col('GPS_DEVICE_KEY').isNull()).count()}")
print(f"Missing Gamma keys: {fact_final.filter(F.col('GAMMA_DETECTOR_KEY').isNull()).count()}")
print(f"Missing Cesium keys: {fact_final.filter(F.col('CESIUM_DETECTOR_KEY').isNull()).count()}")
print(f"Missing Thorium keys: {fact_final.filter(F.col('THORIUM_DETECTOR_KEY').isNull()).count()}")

## Save Tables

In [0]:
# Write tables
gps_dimension.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("GPS_DEVICE")
detector_dimension.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("DETECTOR_DEVICE")
fact_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("MEASUREMENT")

print("Tables saved:")
print(f"GPS_DEVICE: {gps_dimension.count()} rows")
print(f"DETECTOR_DEVICE: {detector_dimension.count()} rows")
print(f"MEASUREMENT: {fact_final.count()} rows")